In [1]:
import io
import os
import time
import requests
import pandas as pd


In [ ]:
import os
import time
import requests
import pandas as pd

# --- edit these per run ---


BASE = "https://rest.uniprot.org/idmapping"
BATCH_SIZE = 500
MASTER_PATH = "/home/callum/sno_data/processed/mapping/uniprot_organisms.csv"
PROCESSED_PATH = "/home/callum/sno_data/processed/"

In [ ]:
def check(resp):
    """Raise on HTTP errors, but print UniProt's explanation first."""
    if not resp.ok:
        print(resp.status_code, resp.text)
    resp.raise_for_status()


def fetch_organisms(ids):
    """Returns (mapping, failed): {input_id: scientific_name} and unmapped IDs."""
    mapping, failed = {}, []

    for start in range(0, len(ids), BATCH_SIZE):
        batch = ids[start:start + BATCH_SIZE]
        batch = [i.strip() for i in batch if i and i.strip() and i.lower() != "nan"]
        if not batch:
            continue

        r = requests.post(
            f"{BASE}/run",
            data={"from": "UniProtKB_AC-ID", "to": "UniProtKB", "ids": ",".join(batch)},
        )
        check(r)
        job_id = r.json()["jobId"]

        while True:
            s = requests.get(f"{BASE}/status/{job_id}")
            check(s)
            job_status = s.json().get("jobStatus")
            if job_status in ("NEW", "RUNNING"):
                time.sleep(1)
                continue
            if job_status is not None:
                raise RuntimeError(f"Job {job_id} ended with status {job_status}")
            break

        res = requests.get(
            f"{BASE}/uniprotkb/results/stream/{job_id}",
            params={"format": "json", "fields": "accession,organism_name"},
        )
        check(res)
        data = res.json()

        for entry in data.get("results", []):
            org = entry["to"].get("organism", {}).get("scientificName")
            if org is None:
                failed.append(entry["from"])
                continue
            mapping[entry["from"]] = org
        failed.extend(data.get("failedIds", []))

    return mapping, failed


def add_organisms(df, id_col="id", out_col="organism", master_path=MASTER_PATH):
    if os.path.exists(master_path):
        master = pd.read_csv(master_path)
    else:
        master = pd.DataFrame(columns=["id", "organism"])

    known = dict(zip(master["id"], master["organism"]))

    ids = df[id_col].dropna().astype(str).unique().tolist()
    new_ids = [i for i in ids if i not in known]
    print(f"{len(known)} already mapped, {len(new_ids)} to query")

    if new_ids:
        new_mapping, failed = fetch_organisms(new_ids)
        known.update(new_mapping)
        if failed:
            print(f"{len(failed)} IDs failed to map, e.g. {failed[:5]}")
        pd.DataFrame(known.items(), columns=["id", "organism"]).to_csv(master_path, index=False)

    df[out_col] = df[id_col].map(known)
    return df

({'P04117': 'Mus musculus', 'P49023': 'Homo sapiens', 'O70250': 'Mus musculus', 'O70251': 'Mus musculus', 'A3KMP2': 'Mus musculus', 'O35490': 'Mus musculus', 'P17427': 'Mus musculus', 'P17426': 'Mus musculus', 'Q8JZN5': 'Mus musculus'}, [])


In [ ]:
from pathlib import Path

DIR_NAME = "ppsno"  # or whichever folder these files live in
in_dir = Path(PROCESSED_PATH) / DIR_NAME

files = [
    "dbsno_neg_processed.csv", "dbsno_pos_processed.csv",
    "independent_neg_processed.csv", "independent_pos_processed.csv",
    "rec_neg_processed.csv", "rec_pos_processed.csv", "rec_processed.csv",
    "reviewed_neg_processed.csv", "reviewed_pos_processed.csv",
]

for f in files:
    df = pd.read_csv(in_dir / f)
    new_df = add_organisms(df)
    out_name = f.replace("_processed.csv", "_org.csv")
    new_df.to_csv(in_dir / out_name, index=False)
    print(f"{f}: {new_df.shape}, {new_df['organism'].isna().sum()} without organism\n")

13 already mapped, 0 to query
        id  position      organism
0   P04117       118  Mus musculus
1   P49023       108  Homo sapiens
2   O70250        23  Mus musculus
3   O70251        50  Mus musculus
4   A3KMP2        24  Mus musculus
5   A3KMP2       124  Mus musculus
6   A3KMP2       363  Mus musculus
7   A3KMP2       425  Mus musculus
8   O35490       131  Mus musculus
9   O35490       217  Mus musculus
10  O35490       256  Mus musculus
11  O35490       299  Mus musculus
12  O35490       300  Mus musculus
13  P17427       330  Mus musculus
14  P17426       283  Mus musculus
15  P17426       941  Mus musculus
16  P17426       970  Mus musculus
17  Q8JZN5       183  Mus musculus
18  Q8JZN5       331  Mus musculus
19  Q8JZN5       617  Mus musculus
